# Project 2 Workbook: LendingClub XAI Professional Model Audit

This is the student workbook for the 2-week XAI project.

Primary dataset: LendingClub loan data.

Your goal is not only to build a strong model, but to audit how it behaves and where it fails.

## Submission Rules (Read First)

This workbook is graded on model quality plus audit quality.

Required:
1. M1 model artifact + validation metrics.
2. M2 local explanations for at least 3 failures (FP/FN).
3. M3 global dependency/interaction evidence.
4. M4 slicing analysis and mitigation proposal.

Coding expectation:
- Cells marked with `TODO REQUIRED` must be implemented by students.
- Do not leave placeholder defaults in required TODO blocks.

Not allowed:
- Plot dumping without interpretation.
- Claiming bias/fairness without subgroup metrics.
- Tuning on test set.

## Milestone Map

- M1: Black Box Model (performance first)
- M2: Local Detective (3 failure explanations)
- M3: Global Audit (importance, PDP, interactions)
- M4: Slicing + Audit Report (where model fails by subgroup and what to fix)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import shap
from lime.lime_tabular import LimeTabularExplainer
from lightgbm import LGBMClassifier
import joblib

RANDOM_SEED = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 140)

## Team Configuration (Fill This)

Set your team metadata and assumptions before modeling.

In [ ]:
team_id = 'TODO_TEAM_ID'
audit_focus = 'TODO: e.g., false negatives in high-income applicants'
threshold = 0.50  # You must justify and can update later using validation set

print('Team:', team_id)
print('Audit focus:', audit_focus)
print('Initial threshold:', threshold)

## Part 1: Load LendingClub Data

Expected file location:
- data/lendingclub_loan_data.csv

If your file name is different, change DATA_FILE below.

In [ ]:
DATA_DIR = Path('data')
DATA_FILE = DATA_DIR / 'lendingclub_loan_data.csv'

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f'Required LendingClub file not found: {DATA_FILE}. Place the CSV in project 2/data.'
    )

df_raw = pd.read_csv(DATA_FILE)
print('Loaded:', DATA_FILE)
print('Shape:', df_raw.shape)
display(df_raw.head(3))

## Part 2: Build Target and Core Features

Target for this workbook:
- `y = 1` for risky outcome (charged off/default/late-style statuses)

TODO (analysis-focused):
1. Inspect and print your dataset-specific `loan_status` values.
2. Complete risky label mapping for your file version.
3. Expand leakage filtering with justified columns.
4. Optional: add engineered features only if they improve explanation clarity.

In [ ]:
df = df_raw.copy()

if 'loan_status' not in df.columns:
    raise ValueError('Expected a loan_status column in LendingClub dataset.')

# Inspect label space and keep this output in your report appendix.
status = df['loan_status'].astype(str).str.lower().str.strip()
display(status.value_counts(dropna=False).head(20))

# Expand risky token list only if your LendingClub file uses additional risky labels.
risky_tokens = [
    'charged off',
    'default',
    # 'late (31-120 days)',
    # 'does not meet the credit policy. status:charged off',
]
y = status.apply(lambda s: int(any(tok in s for tok in risky_tokens)))

# Starter leakage filters: add/remove based on your schema and explain choices in markdown.
leak_tokens = [
    'recover',
    'collection',
    'last_pymnt',
    'next_pymnt',
    'settlement',
]
feature_cols = []
for c in df.columns:
    lc = c.lower()
    if c == 'loan_status':
        continue
    if any(tok in lc for tok in leak_tokens):
        continue
    feature_cols.append(c)

X = df[feature_cols].copy()

print('Positive rate:', round(float(y.mean()), 4))
print('Features:', X.shape[1])
display(X.head(2))

In [ ]:
# Optional: sample rows for faster iteration on student laptops.
MAX_ROWS = 120000
if len(X) > MAX_ROWS:
    idx = np.random.default_rng(RANDOM_SEED).choice(len(X), size=MAX_ROWS, replace=False)
    X = X.iloc[idx].reset_index(drop=True)
    y = y.iloc[idx].reset_index(drop=True)

print('Working rows:', len(X))

## Part 3: Train / Validation / Test Split

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RANDOM_SEED, stratify=y_train_val
)

print('Train:', len(X_train), 'Val:', len(X_val), 'Test:', len(X_test))
print('Rates train/val/test:', round(float(y_train.mean()),4), round(float(y_val.mean()),4), round(float(y_test.mean()),4))

## M1: The Black Box

Deliverable requirements:
1. Train one strong baseline model (provided LightGBM config is enough).
2. Report validation AUC and F1 at your chosen threshold.
3. Save model artifact (`.pkl`).
4. Write 3-5 lines on threshold trade-offs and error types.

Note: Hyperparameter tuning is optional, not required. Prioritize explanation depth in M2-M4.

In [ ]:
numeric_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

# TODO REQUIRED: Build the preprocessing pipeline.
# Hint:
# 1) Numeric pipeline: median imputer
# 2) Categorical pipeline: most_frequent imputer + OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
preprocessor = None  # TODO REQUIRED

# TODO REQUIRED: Fit transform on train, transform on val/test.
X_train_t = None  # TODO REQUIRED
X_val_t = None    # TODO REQUIRED
X_test_t = None   # TODO REQUIRED

feature_names = numeric_cols + categorical_cols

if preprocessor is None or X_train_t is None or X_val_t is None or X_test_t is None:
    raise NotImplementedError('Complete TODO REQUIRED preprocessing block before continuing.')

In [ ]:
# Baseline model config (sufficient for this project).
params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': -1,
    'subsample': 0.9,
    'colsample_bytree': 0.8,
    'random_state': RANDOM_SEED,
}

model = LGBMClassifier(**params)
model.fit(X_train_t, y_train)

val_proba = model.predict_proba(X_val_t)[:, 1]

# TODO REQUIRED: Implement threshold comparison table on validation set.
# Required output columns: threshold, f1, positive_prediction_rate
threshold_grid = [0.3, 0.4, 0.5, 0.6]
threshold_table = None  # TODO REQUIRED
display(threshold_table)

# TODO REQUIRED: choose final threshold using your analysis above.
# Update the `threshold` variable (defined in Team Configuration) if needed.
val_pred = (val_proba >= threshold).astype(int)
print('Validation ROC-AUC:', round(roc_auc_score(y_val, val_proba), 4))
print('Validation F1:', round(f1_score(y_val, val_pred), 4))
print('Confusion matrix (val):')
display(pd.DataFrame(confusion_matrix(y_val, val_pred), index=['actual_0','actual_1'], columns=['pred_0','pred_1']))

if threshold_table is None:
    raise NotImplementedError('Complete TODO REQUIRED threshold_table before finalizing M1.')

In [ ]:
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({'model': model, 'preprocessor': preprocessor, 'feature_names': feature_names}, ARTIFACT_DIR / 'm1_lendingclub_model.pkl')
print('Saved:', ARTIFACT_DIR / 'm1_lendingclub_model.pkl')

## M1 Reflection (Write Here)

- What metric trade-off do you observe?
- Is your current threshold operationally reasonable?
- What is your biggest model limitation at this stage?

## M2: Local Detective

Deliverable requirements:
1. Select at least 3 failure cases (FP/FN).
2. Explain each case with SHAP and/or LIME.
3. Write case-by-case interpretation notes.

In [ ]:
val_results = pd.DataFrame({
    'y_true': y_val.values,
    'y_prob': val_proba,
    'y_pred': val_pred
}, index=X_val.index)

fp_idx = val_results[(val_results['y_true'] == 0) & (val_results['y_pred'] == 1)].index.tolist()
fn_idx = val_results[(val_results['y_true'] == 1) & (val_results['y_pred'] == 0)].index.tolist()

print('False positives:', len(fp_idx))
print('False negatives:', len(fn_idx))
display(val_results.head())

# TODO REQUIRED: Build selected_cases with at least 3 ids including both FP and FN examples.
# Example target structure: [fp_idx[0], fp_idx[1], fn_idx[0]]
selected_cases = []  # TODO REQUIRED

In [ ]:
# SHAP local explanation helper (provided so you can focus on interpretation).
explainer = shap.TreeExplainer(model)

def explain_with_shap(idx):
    pos = X_val.index.get_loc(idx)
    row = X_val_t[pos:pos+1]
    pred = float(model.predict_proba(row)[:, 1][0])

    sv = explainer.shap_values(row)
    if isinstance(sv, list):
        sv = sv[1] if len(sv) > 1 else sv[0]

    print('Case:', idx, 'pred_risk=', round(pred, 4))
    return pd.Series(sv[0], index=feature_names).sort_values(key=np.abs, ascending=False).head(10)

# You should still choose the failure cases yourself to support your audit narrative.
selected_cases = []

if len(selected_cases) < 3:
    raise ValueError('Choose at least 3 failure cases in selected_cases.')

for case_id in selected_cases:
    display(explain_with_shap(case_id))

In [ ]:
# LIME local explanation for one selected case
lime_explainer = LimeTabularExplainer(
    training_data=np.asarray(X_train_t),
    feature_names=feature_names,
    class_names=['safe', 'risky'],
    mode='classification',
    random_state=RANDOM_SEED
)

case_for_lime = selected_cases[0] if selected_cases else X_val.index[0]
case_pos = X_val.index.get_loc(case_for_lime)
exp = lime_explainer.explain_instance(
    data_row=np.asarray(X_val_t[case_pos]),
    predict_fn=model.predict_proba,
    num_features=10
)
print('LIME case:', case_for_lime)
_ = exp.as_pyplot_figure()
plt.tight_layout()
plt.show()

## M2 Reflection (Write Here)

For each of 3 cases, add:
- Why the model failed
- Which features drove the decision
- Whether the explanation seems plausible to domain stakeholders

## M3: Global Audit

Deliverable requirements:
1. Global feature ranking.
2. PDP for top numeric drivers.
3. One interaction finding with evidence.

In [ ]:
perm = permutation_importance(
    estimator=model,
    X=X_val_t,
    y=y_val,
    n_repeats=8,
    random_state=RANDOM_SEED,
    scoring='roc_auc'
)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
display(importance_df.head(15))

plt.figure(figsize=(8, 5))
top = importance_df.head(12).iloc[::-1]
plt.barh(top['feature'], top['importance_mean'])
plt.title('Permutation Importance (Validation)')
plt.xlabel('Mean ROC-AUC Drop')
plt.tight_layout()
plt.show()

In [ ]:
top_numeric = [f for f in importance_df['feature'].tolist() if f in numeric_cols][:2]
if not top_numeric:
    print('No numeric features found for PDP.')
else:
    idxs = [feature_names.index(f) for f in top_numeric]
    fig, ax = plt.subplots(1, len(idxs), figsize=(6 * len(idxs), 4))
    if len(idxs) == 1:
        ax = [ax]
    PartialDependenceDisplay.from_estimator(model, X_val_t, idxs, feature_names=feature_names, ax=ax)
    plt.suptitle('PDP: Top Numeric Features', y=1.03)
    plt.tight_layout()
    plt.show()

In [ ]:
# TODO: Choose two features for interaction analysis and justify your choice in markdown.
candidate_features = importance_df['feature'].head(10).tolist()
print('Top candidates:', candidate_features)

feature_a = None  # TODO set, e.g., 'dti'
feature_b = None  # TODO set, e.g., 'annual_inc'

if feature_a is None or feature_b is None:
    raise ValueError('Set feature_a and feature_b before running interaction analysis.')

if feature_a not in feature_names or feature_b not in feature_names:
    raise ValueError('Chosen features must be present in feature_names.')

i1, i2 = feature_names.index(feature_a), feature_names.index(feature_b)
X_inter = X_val_t[np.random.default_rng(RANDOM_SEED).choice(np.arange(X_val_t.shape[0]), size=min(800, X_val_t.shape[0]), replace=False)]

inter_explainer = shap.TreeExplainer(model)
inter_vals = inter_explainer.shap_interaction_values(X_inter)
if isinstance(inter_vals, list):
    inter_vals = inter_vals[1] if len(inter_vals) > 1 else inter_vals[0]

strength = np.abs(inter_vals[:, i1, i2]).mean()
print(f'SHAP mean |interaction| between {feature_a} and {feature_b}: {strength:.5f}')
shap.dependence_plot(i1, inter_explainer.shap_values(X_inter), X_inter, feature_names=feature_names, interaction_index=i2, show=False)
plt.title(f'SHAP Dependence: {feature_a} colored by {feature_b}')
plt.tight_layout()
plt.show()

## M3 Reflection (Write Here)

- What are the top 3 globally important features?
- Which PDP patterns look non-linear?
- What hidden interaction rule did you find?

## M4: Slicing Analysis and Audit Report

Deliverable requirements:
1. Choose subgroup column(s) and report per-group metrics.
2. Identify one group with significant performance weakness.
3. Propose a practical fix (data vs features vs both).

In [ ]:
slice_candidates = ['home_ownership', 'purpose', 'grade', 'sub_grade', 'emp_length', 'addr_state', 'verification_status']
available = [c for c in slice_candidates if c in X_val.columns]
print('Available slice columns:', available)

# TODO: Manually choose one available slice column and justify choice in markdown.
slice_col = None

if slice_col is None:
    raise ValueError('Set slice_col to one of the available slice columns.')
print('Chosen slice column:', slice_col)

In [ ]:
tmp = pd.DataFrame({
    'group': X_val[slice_col].astype(str).fillna('missing'),
    'y_true': y_val.values,
    'y_prob': val_proba
})
tmp['y_pred'] = (tmp['y_prob'] >= threshold).astype(int)

rows = []
for g, part in tmp.groupby('group'):
    if len(part) < 50:
        continue
    # TODO: compute at least AUC and F1 for each slice; add one extra metric of your choice.
    auc = roc_auc_score(part['y_true'], part['y_prob']) if part['y_true'].nunique() > 1 else np.nan
    f1 = f1_score(part['y_true'], part['y_pred']) if part['y_true'].nunique() > 1 else np.nan
    metric_extra = np.nan  # TODO replace with your chosen metric
    rows.append({
        'group': g,
        'n': len(part),
        'positive_rate': part['y_true'].mean(),
        'auc': auc,
        'f1': f1,
        'metric_extra': metric_extra
    })

slice_report = pd.DataFrame(rows).sort_values(['f1', 'auc'], na_position='last')
display(slice_report.head(20))

## Final Audit Report (Write Here)

Use this exact structure:
1. Executive Summary (5-8 lines)
2. M1 Findings: model quality and threshold rationale
3. M2 Findings: 3 local failure case explanations
4. M3 Findings: global importance, PDP, interaction rule
5. M4 Findings: subgroup failure and risk concentration
6. Recommended Fix: data vs feature vs combined
7. Limitations and Monitoring Plan

## Submission Checklist

- [ ] Workbook completed with all milestone reflections
- [ ] Model artifact saved in artifacts/
- [ ] At least 3 local error explanations included
- [ ] At least 1 interaction effect documented
- [ ] Slicing analysis table included
- [ ] Final audit report completed